# Hybrid KD INT8 Calibration Lock

Khóa 128 ảnh Calibration và 32 ảnh parity holdout từ Karpathy Validation.
Notebook không đọc ảnh, không huấn luyện và không đọc Held-out Test.

## Input bắt buộc — đúng 2 output

1. `HYB_Mobile_Entry`;
2. `01_MSCOCO_Data_Contract`.

Runtime: `CPU / Accelerator None / Internet Off`.

Notebook này chạy **song song** với `HYB_ONNX_FP32`.

In [1]:
import csv
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT", "/kaggle/working/HYB_Calib_Lock"
))
OUT.mkdir(parents=True, exist_ok=True)

CALIBRATION_COUNT = 128
PARITY_COUNT = 32
SELECTION_SALT = "HYB_INT8_VALIDATION_V1"

def now():
    return datetime.now(timezone.utc).isoformat()

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

def exactly_one(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (name, matches)
    return matches[0]

entry_path = exactly_one("HYB_MOBILE_ENTRY_COMPLETE.json")
entry = read_json(entry_path)
assert entry["status"] == "complete"
assert int(entry["selected_seed"]) == 2026
assert entry["test_set_used"] is False

split_path = exactly_one("karpathy_image_split_manifest.csv")
rows = []
with split_path.open("r", encoding="utf-8", newline="") as handle:
    for row in csv.DictReader(handle):
        split_value = row.get("karpathy_split", row.get("split", "")).lower()
        if split_value in {"val", "validation"}:
            image_path = row.get("image_path", row.get("filename", ""))
            rows.append({
                "image_id": int(row["image_id"]),
                "filename": Path(image_path).name,
                "source_partition": row.get("source_partition", ""),
            })
assert len(rows) == 5000
assert len({row["image_id"] for row in rows}) == 5000

def rank_key(row):
    return hashlib.sha256(
        f"{SELECTION_SALT}|{row['image_id']}".encode("utf-8")
    ).hexdigest()

ranked = sorted(rows, key=lambda row: (rank_key(row), row["image_id"]))
calibration = ranked[:CALIBRATION_COUNT]
parity = ranked[CALIBRATION_COUNT:CALIBRATION_COUNT + PARITY_COUNT]
assert {row["image_id"] for row in calibration}.isdisjoint(
    {row["image_id"] for row in parity}
)

csv_path = OUT / "HYB_INT8_COHORTS.csv"
with csv_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "cohort", "ordinal", "image_id", "filename",
            "source_partition", "selection_rank_sha256",
        ],
    )
    writer.writeheader()
    for cohort, cohort_rows in (
        ("calibration", calibration),
        ("parity_holdout", parity),
    ):
        for ordinal, row in enumerate(cohort_rows):
            writer.writerow({
                "cohort": cohort,
                "ordinal": ordinal,
                **row,
                "selection_rank_sha256": rank_key(row),
            })

contract = {
    "schema_version": "hyb_int8_cohort_lock_v1",
    "created_at_utc": now(),
    "status": "locked",
    "selected_seed": 2026,
    "selection_rule": "ascending SHA-256(salt|image_id)",
    "selection_salt": SELECTION_SALT,
    "split": "Karpathy Validation",
    "validation_images": 5000,
    "calibration_count": CALIBRATION_COUNT,
    "parity_holdout_count": PARITY_COUNT,
    "calibration_image_ids": [row["image_id"] for row in calibration],
    "parity_holdout_image_ids": [row["image_id"] for row in parity],
    "cohort_csv": {
        "relative_path": csv_path.name,
        "sha256": sha256(csv_path),
    },
    "source_split_manifest_sha256": sha256(split_path),
    "test_set_status": "sealed",
    "test_set_used": False,
}
contract_path = OUT / "HYB_INT8_COHORT_LOCK.json"
write_json(contract_path, contract)
marker = {
    "schema_version": "hyb_calib_lock_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "contract": {
        "relative_path": contract_path.name,
        "sha256": sha256(contract_path),
    },
    "calibration_count": CALIBRATION_COUNT,
    "parity_holdout_count": PARITY_COUNT,
    "test_set_used": False,
}
write_json(OUT / "HYB_CALIB_LOCK_COMPLETE.json", marker)

print("Calibration lock         : COMPLETE")
print("Validation images        : 5,000")
print("Calibration / parity     : 128 / 32")
print("Held-out Test            : SEALED / NOT READ")

Calibration lock         : COMPLETE
Validation images        : 5,000
Calibration / parity     : 128 / 32
Held-out Test            : SEALED / NOT READ
